In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
        .builder
        .appName("joinsApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.adaptive.enabled", "false")
        .enableHiveSupport()
        .getOrCreate()
)

sc= spark.sparkContext
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/22 15:43:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


25/06/22 15:43:29 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
# check default value

spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'10485760b'

25/06/17 20:55:31 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
# Disable broadcast join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [4]:
yellowTaxiSchema = StructType([
 StructField("VendorID", IntegerType(), True),
 StructField("tpep_pickup_datetime", TimestampType(), True),
 StructField("tpep_dropoff_datetime", TimestampType(), True),
 StructField("passenger_count", DoubleType(), True),
 StructField("trip_distance", DoubleType(), True),
 StructField("RatecodeID", DoubleType(), True),
 StructField("store_and_fwd_flag", StringType(), True),
 StructField("PULocationID", IntegerType(), True),
 StructField("DOLocationID", IntegerType(), True),
 StructField("payment_type", IntegerType(), True),
 StructField("fare_amount", DoubleType(), True),
 StructField("extra", DoubleType(), True),
 StructField("mta_tax", DoubleType(), True),
 StructField("tip_amount", DoubleType(), True),
 StructField("tolls_amount", DoubleType(), True),
 StructField("improvement_surcharge", DoubleType(), True),
 StructField("total_amount", DoubleType(), True),
 StructField("congestion_surcharge", DoubleType(), True),
 StructField("airport_fee", DoubleType(), True),
])

yellowTaxisDf = spark.read.option("header", "true").schema(yellowTaxiSchema).csv(
    "./Files/YellowTaxis_202210.csv"
)

yellowTaxisDf.count()

3675412

In [7]:
taxiZonesSchema = 'PickUpLocationID INT, Borough STRING, Zone STRING, ServiceZone STRING'

taxiZonesDf = spark.read.schema(taxiZonesSchema).csv(
    "./Files/TaxiZones.csv"
)

taxiZonesDf.count()

265

In [8]:
joinedDf = yellowTaxisDf.join(
    taxiZonesDf,
    yellowTaxisDf.PULocationID == taxiZonesDf.PickUpLocationID,
    "inner"
)
joinedDf.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------------+---------+---------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|PickUpLocationID|  Borough|           Zone|ServiceZone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------------+---------+---------------+-----------+
|       1| 2022-10-01 

In [9]:
# Manually broadcast the taxi zones DataFrame
from pyspark.sql.functions import broadcast
joinedDf2 = yellowTaxisDf.join(
    broadcast(taxiZonesDf),
    yellowTaxisDf.PULocationID == taxiZonesDf.PickUpLocationID,
    "inner"
)
joinedDf2.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------------+---------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|PickUpLocationID|  Borough|                Zone|ServiceZone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------------+---------+--------------------+-----------+
|      

In [5]:
# Optimising shuffle sort join by bucketing
# Create 2 views on a large dataset
yellowTaxisDf.createOrReplaceTempView("YellowTaxis1Unbucketed")
yellowTaxisDf.createOrReplaceTempView("YellowTaxis2Unbucketed")

In [6]:
# Join the 2 views on pickup location id of view 1 with drop loaction id of view 2
spark.sql("""
SELECT *
FROM YellowTaxis1Unbucketed AS t1
JOIN YellowTaxis2Unbucketed AS t2
ON t1.PULocationID = t2.DOLocationID
""").show()

25/06/22 15:43:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|to

In [7]:
# Create bucketed tables
yellowTaxisDf.write.bucketBy(4, "PULocationID").option("header", "true").option("dateFormat", "yyyy-MM-dd HH:mm:ss.S").mode("overwrite").format("csv").saveAsTable("YellowTaxisPickUpBucket")
yellowTaxisDf.write.bucketBy(4, "DOLocationID").option("header", "true").option("dateFormat", "yyyy-MM-dd HH:mm:ss.S").mode("overwrite").format("csv").saveAsTable("YellowTaxisDropOffBucket")

25/06/22 15:49:32 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/06/22 15:49:32 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
25/06/22 15:49:33 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
25/06/22 15:49:33 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore i545672@192.168.1.2
25/06/22 15:49:40 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider csv. Persisting data source table `spark_catalog`.`default`.`yellowtaxispickupbucket` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
25/06/22 15:49:40 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
25/06/22 15:49:40 WARN HiveConf: HiveConf of

In [8]:
# Join the 2 bucketed tables
spark.sql("""
SELECT *
FROM YellowTaxisPickUpBucket AS t1
JOIN YellowTaxisDropOffBucket AS t2
ON t1.PULocationID = t2.DOLocationID
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|to

25/06/22 16:10:00 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 950855 ms exceeds timeout 120000 ms
25/06/22 16:10:00 WARN SparkContext: Killing executors is not supported by current scheduler.
25/06/22 16:10:03 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o